In [30]:
import pandas as pd
import numpy as np
import sqlite3


In [31]:
conn = sqlite3.connect("C:/Users/abhis/OneDrive/Documents/abhishek/Data_Analyst_Abhishek/data_analyst/python_data/practice/enterprise ecommerce project/database/managing.db")

In [ ]:
pd.read_sql_query('select * from customers limit 3', conn)

In [33]:
vendors = pd.read_csv("C:/Users/abhis/OneDrive/Documents/abhishek/Data_Analyst_Abhishek/data_analyst/python_data/practice/enterprise ecommerce project/data/vendors.csv")

In [ ]:
vendors.head(4)

In [ ]:
vendors.to_sql("vendors", conn, if_exists='replace', index=False)

In [36]:
# python cleaning 
# vendor_name     5
# rating         16

In [37]:
vendors = vendors.drop_duplicates(subset='vendor_id', keep='first').reset_index(drop= True)

In [38]:
# clean vendor name
vendors['vendor_name'] = vendors['vendor_name'].str.strip().str.title()

In [39]:
vendors = vendors.dropna(subset='vendor_name')

In [40]:
vendors['country'] = vendors['country'].str.strip().str.title()
vendors['vendor_id'] = vendors['vendor_id'].str.strip().str.lower()


In [ ]:
vendors.info()

In [42]:
vendors['rating'] = vendors['rating'].fillna(vendors.groupby('country')['rating'].transform('median'))

In [ ]:
vendors.info()

In [ ]:
vendors['rating'].max()

In [53]:
# remove outliers 
vendors = vendors[(
    vendors['rating'] >= 1
) & (
    vendors['rating'] <= 5
)]

In [ ]:
vendors.info()

In [ ]:
vendors['rating'].max()

In [ ]:
vendors

In [59]:
# # SQL Questions 
# Easy
# Count vendors by country.
# Average vendor rating.
# Vendors with missing ratings.
# Medium
# Top 10 highest-rated vendors.
# Vendor distribution by country.
# Duplicate vendor names.
# Vendors with invalid ratings.
# Hard
# Standardize vendor names using SQL string functions.
# Detect fuzzy duplicates (after trimming and changing case).
# Rank vendors by rating within each country.
# Find countries where average vendor rating is below the global average.

In [ ]:
# Count vendors by country.
pd.read_sql_query('select country, count(distinct vendor_id) as total_vendor from vendors group by country order by total_vendor', conn)

In [ ]:
# Average vendor rating.
pd.read_sql_query('with valid_raitng as (select * from vendors where rating >= 1 and rating <=5) select vendor_name, avg(rating) as avg_rating from valid_raitng group by vendor_name order by avg_rating desc', conn)


In [ ]:
# Vendors with missing ratings.
pd.read_sql_query('select vendor_id, vendor_name, rating from vendors where rating is null', conn)

In [ ]:
# Top 10 highest-rated vendors.
pd.read_sql_query('with valid_raitng as (select * from vendors where rating >= 1 and rating <=5), rnk_data as (select vendor_name , rating, dense_rank() over(order by rating desc) as rnk from valid_raitng) select vendor_name, rating from rnk_data where rnk <= 10 limit 10', conn)

In [ ]:
# Vendor distribution by country.
pd.read_sql_query('select country, count(vendor_id) as total_vendors from vendors group by country order by total_vendors desc' , conn)

In [ ]:
# Duplicate vendor names.
pd.read_sql_query('select vendor_name, count(vendor_name) as dupp_value from vendors group by vendor_name having count(vendor_name) > 1 order by dupp_value desc', conn)

In [ ]:
# Vendors with invalid ratings.
pd.read_sql_query('select vendor_id, vendor_name , rating from vendors where rating is null or rating > 5 or rating < 1', conn)

In [ ]:
# Standardize vendor names using SQL string functions.
pd.read_sql_query("select trim(lower(vendor_name)) as vendor_name from vendors ", conn)

In [ ]:
# Detect fuzzy duplicates (after trimming and changing case).
pd.read_sql_query('with clean_name as (select trim(lower(vendor_name)) as vendor_name from vendors) select vendor_name, count(vendor_name) as dupp from clean_name group by vendor_name having count(vendor_name) > 1 ', conn)


In [ ]:
# Rank vendors by rating within each country.
pd.read_sql_query('select country, vendor_name, rating, dense_rank() over(partition by country order by rating desc) as rating_rank from vendors ', conn)